In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Qiskit Function oleh Q-CTRL Fire Opal
*Lihat [referensi API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (melalui IBM Quantum Platform API) Plan. Fitur ini dalam status rilis pratinjau dan bisa berubah sewaktu-waktu.


<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Ikhtisar
Dengan Fire Opal Optimization Solver, kamu bisa memecahkan masalah optimasi skala utilitas di perangkat keras kuantum tanpa perlu keahlian kuantum. Cukup masukkan definisi masalah tingkat tinggi, dan Solver akan menangani sisanya. Seluruh alur kerja bersifat noise-aware dan memanfaatkan [Fire Opal's Performance Management](/guides/q-ctrl-performance-management) di balik layar. Solver secara konsisten menghasilkan solusi yang akurat untuk masalah yang menantang secara klasik, bahkan pada skala perangkat penuh di QPU IBM&reg; terbesar.

Solver bersifat fleksibel dan dapat digunakan untuk memecahkan masalah optimasi kombinatorial yang didefinisikan sebagai fungsi objektif atau graf sembarang. Masalah tidak perlu dipetakan ke topologi perangkat. Baik masalah tidak terkendali maupun terkendali dapat diselesaikan, asalkan kendala dapat dirumuskan sebagai suku penalti. Contoh-contoh yang disertakan dalam panduan ini menunjukkan cara memecahkan masalah optimasi skala utilitas yang tidak terkendali dan terkendali menggunakan berbagai tipe input Solver. Contoh pertama melibatkan masalah Max-Cut yang didefinisikan pada graf 3-Regular dengan 156 simpul, sementara contoh kedua menangani masalah Minimum Vertex Cover 50 simpul yang didefinisikan oleh fungsi biaya.

Untuk mendapatkan akses ke Optimization Solver, [hubungi Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Deskripsi fungsi
Solver sepenuhnya mengoptimalkan dan mengotomatiskan seluruh algoritma, mulai dari supresi kesalahan di tingkat perangkat keras hingga pemetaan masalah yang efisien dan optimasi klasik loop tertutup. Di balik layar, pipeline Solver mengurangi kesalahan di setiap tahap, memungkinkan peningkatan performa yang dibutuhkan untuk penskalaan yang bermakna. Alur kerja yang mendasarinya terinspirasi dari Quantum Approximate Optimization Algorithm (QAOA), yang merupakan algoritma hibrida kuantum-klasik. Untuk ringkasan lengkap alur kerja Optimization Solver, lihat [manuskrip yang dipublikasikan](https://arxiv.org/abs/2406.01743).

![Visualisasi alur kerja Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Untuk memecahkan masalah umum dengan Optimization Solver:
1. Definisikan masalahmu sebagai fungsi objektif, graf, atau rantai spin `SparsePauliOp`.
2. Hubungkan ke fungsi melalui Qiskit Functions Catalog.
3. Jalankan masalah dengan Solver dan ambil hasilnya.
### Format masalah yang diterima
- Representasi ekspresi polinomial dari fungsi objektif. Idealnya dibuat di Python dengan objek SymPy Poly yang sudah ada dan diformat menjadi string menggunakan [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Representasi graf dari tipe masalah tertentu. Graf harus dibuat menggunakan library networkx di Python. Kemudian dikonversi ke string menggunakan fungsi networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Representasi rantai spin dari masalah tertentu. Rantai spin harus direpresentasikan sebagai objek `SparsePauliOp`; lihat [dokumentasi](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) untuk detail lebih lanjut.

> **Note:** Jika kamu ingin menggunakan Backend yang belum didukung oleh fungsi ini, [hubungi Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) untuk menambahkan dukungan.
## Tolok ukur
[Hasil tolok ukur yang dipublikasikan](https://arxiv.org/abs/2406.01743) menunjukkan bahwa Solver berhasil memecahkan masalah dengan lebih dari 120 qubit, bahkan mengungguli hasil yang sebelumnya dipublikasikan pada anil kuantum dan perangkat ion terjebak. Metrik tolok ukur berikut memberikan indikasi kasar tentang akurasi dan penskalaan tipe masalah berdasarkan beberapa contoh. Metrik aktual bisa berbeda berdasarkan berbagai fitur masalah, seperti jumlah suku dalam fungsi objektif (kepadatan) dan lokalitasnya, jumlah variabel, dan orde polinomial.

"Jumlah qubit" yang ditunjukkan bukanlah batasan keras tetapi merepresentasikan ambang batas kasar di mana kamu dapat mengharapkan akurasi solusi yang sangat konsisten. Ukuran masalah yang lebih besar telah berhasil dipecahkan, dan pengujian di luar batas ini sangat dianjurkan.

Konektivitas qubit sembarang didukung di semua tipe masalah.

| Tipe masalah    | Jumlah qubit | Contoh | Akurasi | Total waktu (s) | Penggunaan runtime (s) | Jumlah iterasi
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Masalah kuadratik dengan koneksi jarang  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Optimasi biner orde tinggi | 156 | Model spin-glass Ising | 100%      | 1461     | 272          | 16 |
| Masalah kuadratik dengan koneksi padat | 50 | Max-Cut penuh terkoneksi| 100%      |  1758    | 268  | 12 |
| Masalah terkendali dengan suku penalti | 50 | Weighted Minimum Vertex Cover dengan kepadatan tepi 8% | 100%      | 1074     | 215 | 10 |
## Mulai
Pertama, autentikasi menggunakan [kunci API IBM Quantum](http://quantum.cloud.ibm.com/)-mu. Kemudian, pilih Qiskit Function sebagai berikut. (Cuplikan ini mengasumsikan kamu sudah [menyimpan akunmu](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokalmu.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Definisikan masalah
Kamu bisa menjalankan masalah Max-Cut dengan mendefinisikan masalah graf dan menentukan `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [5]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [6]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Jalankan masalah
Saat menggunakan metode input berbasis graf, tentukan tipe masalah.

In [7]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [8]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [9]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [10]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

ba5fbdb8-9e71-49bd-8320-79dcdb46de6a
QUEUED


### 3. Ambil hasilnya
Ambil nilai potongan optimal dari kamus hasil.

> **Note:** Pemetaan variabel ke bitstring mungkin telah berubah. Kamus output berisi sub-kamus `variables_to_bitstring_index_map`, yang membantu memverifikasi urutannya.

In [11]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [12]:
# %pip install numpy networkx sympy

Kamu bisa memverifikasi akurasi hasil dengan memecahkan masalah secara klasik menggunakan solver open-source seperti [PuLP](https://coin-or.github.io/pulp/) jika graf tidak terhubung padat. Masalah dengan kepadatan tinggi mungkin memerlukan solver klasik tingkat lanjut untuk memvalidasi solusinya.
## Contoh: Optimasi terkendali
Contoh Max-Cut sebelumnya adalah masalah optimasi biner kuadratik tidak terkendali yang umum. Optimization Solver Q-CTRL dapat digunakan untuk berbagai tipe masalah, termasuk optimasi terkendali. Kamu bisa memecahkan tipe masalah sembarang dengan memasukkan definisi masalah yang direpresentasikan sebagai polinomial di mana kendala dimodelkan sebagai suku penalti.

Contoh berikut mendemonstrasikan cara membangun fungsi biaya untuk masalah optimasi terkendali, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Selain paket `qiskit-ibm-catalog` dan `qiskit`, kamu juga akan menggunakan paket berikut untuk menjalankan contoh ini: `numpy`, `networkx`, dan `sympy`. Kamu bisa menginstal paket-paket ini dengan membatalkan komentar sel berikut jika kamu menjalankan contoh ini di notebook menggunakan kernel IPython.

In [13]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Definisikan masalah
Definisikan masalah MVC acak dengan membuat graf dengan simpul berbobot acak.

In [14]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

![Output of the previous code cell](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Model optimasi standar untuk MVC berbobot dapat dirumuskan sebagai berikut. Pertama, penalti harus ditambahkan untuk setiap kasus di mana sebuah tepi tidak terhubung ke simpul dalam subset. Oleh karena itu, misalkan $n_i = 1$ jika simpul $i$ ada dalam cover (yaitu, dalam subset) dan $n_i = 0$ jika tidak. Kedua, tujuannya adalah meminimalkan jumlah total simpul dalam subset, yang dapat direpresentasikan oleh fungsi berikut:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [15]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

Sekarang setiap tepi dalam graf harus menyertakan setidaknya satu titik ujung dari cover, yang dapat diekspresikan sebagai ketidaksamaan:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Setiap kasus di mana sebuah tepi tidak terhubung ke simpul cover harus dikenai penalti. Ini dapat direpresentasikan dalam fungsi biaya dengan menambahkan penalti berbentuk $P(1-n_i-n_j+n_i n_j)$ di mana $P$ adalah konstanta penalti positif. Dengan demikian, alternatif tidak terkendali untuk ketidaksamaan terkendali pada MVC berbobot adalah:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [16]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. Jalankan masalah

In [17]:
print(mvc_job.status())

QUEUED


Periksa [status](/guides/functions#check-job-status) workload Qiskit Function-mu atau ambil [hasil](/guides/functions#retrieve-results) sebagai berikut:

In [18]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>